In [ ]:
library("sscVis")
library("data.table")
library("grid")
library("cowplot")
library("ggrepel")
library("readr")
library("plyr")
library("ggpubr")
library("tidyverse")
library(viridis)
library(Seurat)
library(pheatmap)

In [ ]:
library(pheatmap)
library(circlize)
library(ComplexHeatmap)

In [ ]:
library(pdftools)

In [ ]:
install.packages("pdftools")

In [ ]:
do.tissueDist <- function(cellInfo.tb = cellInfo.tb,
                          meta.cluster = cellInfo.tb$meta.cluster,
                          colname.patient = "patient",
                          loc = cellInfo.tb$loc,
                          out.prefix,
                          pdf.width=3,
                          pdf.height=5,
                          verbose=0){
  ##input data 
  library(data.table)
  dir.create(dirname(out.prefix),F,T)

  cellInfo.tb = data.table(cellInfo.tb)
  cellInfo.tb$meta.cluster = as.character(meta.cluster)

  if(is.factor(loc)){
    cellInfo.tb$loc = loc
  }else{cellInfo.tb$loc = as.factor(loc)}

  loc.avai.vec <- levels(cellInfo.tb[["loc"]])
  count.dist <- unclass(cellInfo.tb[,table(meta.cluster,loc)])[,loc.avai.vec]
  freq.dist <- sweep(count.dist,1,rowSums(count.dist),"/")
  freq.dist.bin <- floor(freq.dist * 100 / 10)
  print(freq.dist.bin)

  {
    count.dist.melt.ext.tb <- test.dist.table(count.dist)
    p.dist.tb <- dcast(count.dist.melt.ext.tb,rid~cid,value.var="p.value")
    OR.dist.tb <- dcast(count.dist.melt.ext.tb,rid~cid,value.var="OR")
    OR.dist.mtx <- as.matrix(OR.dist.tb[,-1])
    rownames(OR.dist.mtx) <- OR.dist.tb[[1]]
  }

  sscVis::plotMatrix.simple(OR.dist.mtx,
                            out.prefix=sprintf("%s.OR.dist",out.prefix),
                            show.number=F,
                            waterfall.row=T,par.warterfall = list(score.alpha = 2,do.norm=T),
                            exp.name=expression(italic(OR)),
                            z.hi=4,
                            palatte=viridis::viridis(7),
                            pdf.width = 4, pdf.height = pdf.height)
  if(verbose==1){
    return(list("count.dist.melt.ext.tb"=count.dist.melt.ext.tb,
                "p.dist.tb"=p.dist.tb,
                "OR.dist.tb"=OR.dist.tb,
                "OR.dist.mtx"=OR.dist.mtx))
  }else{
    return(OR.dist.mtx)
  }
}

test.dist.table <- function(count.dist,min.rowSum=0)
{
  count.dist <- count.dist[rowSums(count.dist)>=min.rowSum,,drop=F]
  sum.col <- colSums(count.dist)
  sum.row <- rowSums(count.dist)
  count.dist.tb <- as.data.frame(count.dist)
  setDT(count.dist.tb,keep.rownames=T)
  count.dist.melt.tb <- melt(count.dist.tb,id.vars="rn")
  colnames(count.dist.melt.tb) <- c("rid","cid","count")
  count.dist.melt.ext.tb <- as.data.table(ldply(seq_len(nrow(count.dist.melt.tb)), function(i){
    this.row <- count.dist.melt.tb$rid[i]
    this.col <- count.dist.melt.tb$cid[i]
    this.c <- count.dist.melt.tb$count[i]
    other.col.c <- sum.col[this.col]-this.c
    this.m <- matrix(c(this.c,
                       sum.row[this.row]-this.c,
                       other.col.c,
                       sum(sum.col)-sum.row[this.row]-other.col.c),
                     ncol=2)
    res.test <- fisher.test(this.m)
    data.frame(rid=this.row,
               cid=this.col,
               p.value=res.test$p.value,
               OR=res.test$estimate)
  }))
  count.dist.melt.ext.tb <- merge(count.dist.melt.tb,count.dist.melt.ext.tb,
                                  by=c("rid","cid"))
  count.dist.melt.ext.tb[,adj.p.value:=p.adjust(p.value,"BH")]
  return(count.dist.melt.ext.tb)
}

In [ ]:
project_ID="pool"
group_by="leiden_scVI"
mtx="transcriptome"

In [ ]:
meta  <- read.csv(file = paste0(mtx,"/",project_ID, "_DEG/", "meta_treatment_anno.csv"),row.names=1)

In [ ]:

meta$loc <- meta$treatment
meta$meta.cluster <- meta$anno_sub
# 指定输出文件路径及前缀
out.prefix <- "./Fig_OR"

#主分析函数
OR.immune.list <- do.tissueDist(cellInfo.tb=meta,
                                out.prefix=sprintf("%s.Immune_cell",out.prefix),
                                pdf.width=4,pdf.height=8,verbose=1
)

In [ ]:
OR_df = as.data.frame(OR.immune.list[["OR.dist.tb"]])
row.names(OR_df) = OR_df$rid
pval_df = as.data.frame(OR.immune.list[["p.dist.tb"]])
row.names(pval_df) = pval_df$rid

In [ ]:
OR_df <- OR_df[2:dim(OR_df)[2]]
pval_df <- pval_df[2:dim(pval_df)[2]]

In [ ]:
pdf(paste0(mtx, "/",project_ID, "_DEG/","anno_sub_Treatment",".pdf"),width=3, height =6, pointsize = 4)
#pdf(paste0("anno_sub2_treatment",".pdf"),width=2.6, height =5, pointsize = 4)
pheatmap(as.matrix(OR_df)
                   ,fontsize=8, cluster_cols =F, 
                   #annotation_col=meta,
                   #annotation_colors = ann_colors,
          clustering_method = "ward.D2",
         color=colorRamp2(c(0,1,2),c("royalblue","papayawhip","tomato")),
         display_numbers = as.matrix(ifelse(pval_df<0.05, "*",""),nrow=nrow(pval_df)),
         fontsize_number = 12,
         number_format = "%.2f",
                   show_colnames =T )#, scale="column")
dev.off()